# Notebook 17 - Label Naive + Sample Weighting (Soft Deadband) + Random Search Baru

**Notebook 13 TIDAK diubah.** Label yang dipakai TETAP skema naive
(`Close[t+5] > Close[t]`, SEMUA data - tidak ada yang dibuang seperti
triple-barrier di Notebook 16). Yang beda: tiap sampel diberi **bobot loss**
sesuai besar `|forward return 5-hari|` - sampel dengan pergerakan kecil
(dekat 0%, ambigu) diberi bobot rendah, sampel dengan pergerakan besar
(sinyal jelas) dibobot penuh. Ini "soft deadband" yang didiskusikan
sebelumnya: tidak ada data yang dibuang sama sekali, tapi gradient dari
sampel paling ambigu tidak lagi disamakan kuat dengan sampel yang jelas
arahnya.

```
weight_i = clip(|fwd_return_5hari_i| / THETA_REF, W_MIN, 1.0)
```

**Perbedaan penting dari Notebook 16**: kali ini **Random Search dijalankan
ULANG DARI NOL** (20 iterasi, search space "fixed" yang sama dengan
Notebook 13 Bagian 9b) di bawah loss yang sudah diberi bobot -
`best_config` LAMA (dituning untuk loss TANPA bobot) tidak diasumsikan
masih optimal begitu saja untuk loss yang sudah berubah bentuknya.

## Setup

In [ ]:
from __future__ import annotations

import json
import logging
import sys
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("E:/forecasting_index_snp500/stock_prediction")
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("notebook17")

ARTIFACT_DIR = ROOT / "data/artifacts/lstm_weights"
NAIVE_METADATA_PATH = ARTIFACT_DIR / "notebook13_fixed_metadata.json"  # HANYA DIBACA, utk pembanding akhir

OUT_DIR = ROOT / "notebooks" / "outputs" / "notebook17"
OUT_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = 20
HORIZON = 5
SEED = 42
ENSEMBLE_SEEDS = [2, 16, 25, 27, 32, 64]  # SAMA dgn Notebook 13 - apple-to-apple utk ensemble akhir

THETA_REF = 0.01   # forward return >= 1% dianggap "sinyal jelas" -> weight penuh
W_MIN = 0.2        # forward return ~0% tetap dipakai, tapi bobot loss diturunkan ke 0.2 (bukan 0/dibuang)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info("Device: %s", device)

from pipeline.feature_engineering.dataset_builder import load_return_features, RETURN_FEATURE_COLS_B
from models.lstm.notebook07_common import (
    load_split_capped, compute_iqr_bounds, apply_capping, _make_windows,
    compute_class_weights, _evaluate_epoch, sweep_best_threshold, sample_config,
)
from models.lstm.lstm_model import FlexibleLSTMClassifier

FEATURE_COLS = RETURN_FEATURE_COLS_B
CAPPING_COLS = RETURN_FEATURE_COLS_B

with open(NAIVE_METADATA_PATH, encoding="utf-8") as f:
    naive_metadata = json.load(f)
assert naive_metadata["feature_cols"] == FEATURE_COLS
assert naive_metadata["window"] == WINDOW and naive_metadata["horizon"] == HORIZON
print("Baseline pembanding (Notebook 13, TANPA sample weighting):", naive_metadata["metrics"])

## Bagian 1 - Load Data + Label Naive (REUSE `load_split_capped`, Modul Produksi TIDAK Diubah)

`fwd_return_h5` (besar pergerakan, bukan cuma arahnya) dihitung SEBELUM
masuk `load_split_capped()` supaya ikut ter-split kronologis dengan benar
(bukan dihitung ulang per-split yang berisiko salah di batas split).

In [ ]:
def load_data_fn() -> pd.DataFrame:
    df = load_return_features().copy()
    df["fwd_return_h5"] = df["Close"].shift(-HORIZON) / df["Close"] - 1.0
    return df

df_train, df_val, df_test, capping_bounds, n_capped = load_split_capped(
    load_data_fn, capping_cols=CAPPING_COLS, horizon=HORIZON, train_ratio=0.70, val_ratio=0.15,
)

print("Split sizes:", {"train": len(df_train), "val": len(df_val), "test": len(df_test)})
print("mean label  :", {"train": df_train["label"].mean(), "val": df_val["label"].mean(), "test": df_test["label"].mean()})
df_train[["Date", "Close", "fwd_return_h5", "label"]].head()

## Bagian 2 - Hitung Sample Weight + Scaling + Windowing

Weight dihitung dari `fwd_return_h5` (per baris, SEBELUM windowing), lalu
di-window dengan penjajaran indeks YANG SAMA dengan label (`_make_windows`
tidak bisa dipakai langsung untuk weight float karena fungsi itu
memaksa cast label ke `int64` - jadi weight disejajarkan manual dengan
irisan indeks yang identik).

In [ ]:
for d in (df_train, df_val, df_test):
    d["sample_weight"] = (d["fwd_return_h5"].abs() / THETA_REF).clip(upper=1.0).clip(lower=W_MIN)

print("Statistik sample_weight (train):")
print(df_train["sample_weight"].describe())

scaler = MinMaxScaler()
train_feat = scaler.fit_transform(df_train[FEATURE_COLS].values.astype(np.float64))
val_feat = scaler.transform(df_val[FEATURE_COLS].values.astype(np.float64))
test_feat = scaler.transform(df_test[FEATURE_COLS].values.astype(np.float64))

X_train, y_train = _make_windows(train_feat, df_train["label"].values, WINDOW)
X_val, y_val = _make_windows(val_feat, df_val["label"].values, WINDOW)
X_test, y_test = _make_windows(test_feat, df_test["label"].values, WINDOW)


def window_align_1d(values: np.ndarray, window: int, num_samples: int) -> np.ndarray:
    """Sejajarkan array 1D (mis. sample_weight) dgn irisan indeks IDENTIK
    yg dipakai _make_windows utk label - TANPA cast ke int64."""
    return values[window - 1: window - 1 + num_samples].astype(np.float32)


w_train = window_align_1d(df_train["sample_weight"].values, WINDOW, len(y_train))
w_val = window_align_1d(df_val["sample_weight"].values, WINDOW, len(y_val))
w_test = window_align_1d(df_test["sample_weight"].values, WINDOW, len(y_test))

print(f"\nX_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}")
print(f"mean weight -> train={w_train.mean():.4f}  val={w_val.mean():.4f}  test={w_test.mean():.4f}")

## Bagian 3 - Fungsi Training BERBOBOT (Weighted Loss)

`train_one_epoch_weighted` mereplikasi `_train_one_epoch` produksi
(`notebook07_common.py`, TIDAK diubah), HANYA menambahkan perkalian bobot
per-sampel sebelum di-rata-rata (`reduction="none"` lalu weighted mean,
BUKAN mean biasa). Validasi (`_evaluate_epoch`, `val_f1_macro_quick`) TETAP
TANPA bobot - evaluasi harus mencerminkan performa nyata di semua sampel
secara adil, bobot hanya memengaruhi SINYAL GRADIEN saat training.

In [ ]:
def train_one_epoch_weighted(model, loader, optimizer, class_weights, device):
    model.train()
    ce = nn.CrossEntropyLoss(weight=class_weights, reduction="none")
    total_loss, n_samples = 0.0, 0
    y_true_all, y_pred_all = [], []
    for xb, yb, wb in loader:
        xb, yb, wb = xb.to(device), yb.to(device), wb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        per_sample_loss = ce(logits, yb)
        loss = (per_sample_loss * wb).sum() / wb.sum()
        loss.backward()
        optimizer.step()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs
        preds = logits.argmax(dim=1).detach().cpu().numpy()
        y_true_all.extend(yb.detach().cpu().numpy().tolist())
        y_pred_all.extend(preds.tolist())

    avg_loss = total_loss / n_samples if n_samples else 0.0
    acc = accuracy_score(y_true_all, y_pred_all) if n_samples else 0.0
    return avg_loss, float(acc)


def val_f1_macro_quick(model, X_val_t, y_val, n_steps: int = 39) -> float:
    model.eval()
    with torch.no_grad():
        p = torch.softmax(model(X_val_t), dim=1)[:, 1].cpu().numpy()
    best = -1.0
    for t in np.linspace(0.05, 0.95, n_steps):
        best = max(best, f1_score(y_val, (p >= t).astype(int), average="macro", zero_division=0))
    return float(best)


def build_optimizer_wd(name: str, params, lr: float, weight_decay: float = 0.0):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "rmsprop":
        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Optimizer tidak dikenal: {name}")


MIN_EPOCHS = 20   # sama dgn resep "fixed" Notebook 13 Bagian 9b
PATIENCE = 15


def train_with_early_stopping_weighted(model, train_loader, val_loader, X_val_t, y_val,
                                       optimizer, class_weights, device, max_epochs: int,
                                       min_epochs: int = MIN_EPOCHS, patience: int = PATIENCE,
                                       log_prefix: str = "", log_every: int = 50):
    best_score, best_state, epochs_no_improve, stopped_epoch = -1.0, None, 0, max_epochs
    history = {"train_loss": [], "val_f1_macro": []}
    for epoch in range(1, max_epochs + 1):
        train_loss, _ = train_one_epoch_weighted(model, train_loader, optimizer, class_weights, device)
        val_f1m = val_f1_macro_quick(model, X_val_t, y_val)
        history["train_loss"].append(train_loss)
        history["val_f1_macro"].append(val_f1m)
        if epoch == 1 or epoch % log_every == 0 or epoch == max_epochs:
            logger.info("%sepoch %3d/%d - train_loss(weighted)=%.4f val_f1_macro=%.4f", log_prefix, epoch, max_epochs, train_loss, val_f1m)
        if val_f1m > best_score + 1e-5:
            best_score = val_f1m
            best_state = deepcopy({k: v.detach().cpu().clone() for k, v in model.state_dict().items()})
            epochs_no_improve = 0
        else:
            if epoch >= min_epochs:
                epochs_no_improve += 1
            if epoch >= min_epochs and epochs_no_improve >= patience:
                stopped_epoch = epoch
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history, stopped_epoch

## Bagian 4 - Random Search BARU (20 Iterasi, Loss Berbobot)

Search space "fixed" SAMA dengan Notebook 13 Bagian 9b (sudah terbukti
mencegah collapse: `min_epochs=20`, `patience=15`, dropout kecil, lr lebih
tinggi, no `sigmoid`) - HANYA loss training-nya yang beda (berbobot).
`best_config` di sini **TIDAK diwarisi** dari Notebook 13/16.

In [ ]:
SEARCH_SPACE_FIXED = {
    "hidden_size": [16, 32, 64],
    "num_layers": [1, 2],
    "dropout": (0.0, 0.3),
    "optimizer": ["adam", "rmsprop"],
    "activation": ["relu", "tanh"],
    "lr": (5e-4, 5e-3),
    "epochs": [150],
    "batch_size": [32, 64],
}
WEIGHT_DECAY_CHOICES = [0.0, 1e-5]
N_ITER_FIXED = 20

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train), torch.from_numpy(w_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
X_val_t = torch.from_numpy(X_val).to(device)

rng = np.random.default_rng(SEED)
best = {"val_f1_macro": -1.0, "threshold": 0.5, "config": None}
search_history = []

for i in range(N_ITER_FIXED):
    config_i = sample_config(rng, SEARCH_SPACE_FIXED)
    config_i["weight_decay"] = float(rng.choice(WEIGHT_DECAY_CHOICES))

    torch.manual_seed(SEED + i)
    model_i = FlexibleLSTMClassifier(
        input_size=len(FEATURE_COLS), hidden_size=config_i["hidden_size"],
        num_layers=config_i["num_layers"], dropout=config_i["dropout"],
        activation=config_i["activation"], num_classes=2,
    ).to(device)
    optimizer_i = build_optimizer_wd(config_i["optimizer"], model_i.parameters(), config_i["lr"], config_i["weight_decay"])
    class_weights_i = compute_class_weights(y_train, device)

    train_loader_i = DataLoader(train_ds, batch_size=config_i["batch_size"], shuffle=True)
    val_loader_i = DataLoader(val_ds, batch_size=config_i["batch_size"], shuffle=False)

    history_i, stopped_epoch_i = train_with_early_stopping_weighted(
        model_i, train_loader_i, val_loader_i, X_val_t, y_val,
        optimizer_i, class_weights_i, device,
        max_epochs=config_i["epochs"], log_prefix=f"  [iter {i + 1}/{N_ITER_FIXED}] ",
    )

    model_i.eval()
    with torch.no_grad():
        val_probs_i = torch.softmax(model_i(X_val_t), dim=1)[:, 1].cpu().numpy()
    threshold_i, val_f1_macro_i = sweep_best_threshold(y_val, val_probs_i)
    min_train_loss_i = float(min(history_i["train_loss"]))

    is_best = val_f1_macro_i > best["val_f1_macro"]
    search_history.append({"iteration": i + 1, "val_f1_macro": round(val_f1_macro_i, 4),
                            "stopped_epoch": stopped_epoch_i, "min_train_loss": round(min_train_loss_i, 4),
                            "is_new_best": is_best, **config_i})
    logger.info("Iterasi %d/%d - val_f1_macro=%.4f (stop epoch %d/%d, min_train_loss=%.4f)%s",
                i + 1, N_ITER_FIXED, val_f1_macro_i, stopped_epoch_i, config_i["epochs"], min_train_loss_i,
                " (BEST)" if is_best else "")
    if is_best:
        best = {"val_f1_macro": val_f1_macro_i, "threshold": threshold_i, "config": config_i}

best_config = best["config"]
df_search_history = pd.DataFrame(search_history)
collapse_value = f1_score(y_val, np.ones_like(y_val), average="macro", zero_division=0)
n_collapse = int((df_search_history["val_f1_macro"] - collapse_value).abs().lt(1e-3).sum())

print("\nBest Configuration (Random Search BARU, loss berbobot):")
for k, v in best_config.items():
    print(f"  {k:14s}: {v}")
print(f"\nbest_val_f1_macro : {best['val_f1_macro']:.6f}")
print(f"Iterasi collapse ke 1 kelas (val_f1_macro={collapse_value:.4f}): {n_collapse}/{N_ITER_FIXED}")
df_search_history

## Bagian 5 - Training Ensemble Final (6 Seed, `best_config` Baru + Loss Berbobot)

In [ ]:
y_test_arr = y_test.astype(int)
val_loader_final = DataLoader(val_ds, batch_size=best_config["batch_size"], shuffle=False)
X_test_t = torch.from_numpy(X_test).to(device)

seed_runs = []
for seed in ENSEMBLE_SEEDS:
    torch.manual_seed(seed)
    model_s = FlexibleLSTMClassifier(
        input_size=len(FEATURE_COLS), hidden_size=best_config["hidden_size"],
        num_layers=best_config["num_layers"], dropout=best_config["dropout"],
        activation=best_config["activation"], num_classes=2,
    ).to(device)
    optimizer_s = build_optimizer_wd(best_config["optimizer"], model_s.parameters(), best_config["lr"], best_config["weight_decay"])
    class_weights_s = compute_class_weights(y_train, device)
    train_loader_s = DataLoader(train_ds, batch_size=best_config["batch_size"], shuffle=True)

    history_s, stopped_epoch_s = train_with_early_stopping_weighted(
        model_s, train_loader_s, val_loader_final, X_val_t, y_val,
        optimizer_s, class_weights_s, device, max_epochs=best_config["epochs"],
        log_prefix=f"  [seed {seed}] ",
    )
    seed_runs.append({"seed": seed, "model": model_s, "stopped_epoch": stopped_epoch_s,
                       "min_train_loss": float(min(history_s["train_loss"]))})
    logger.info("seed=%d done (stop epoch %d, min_train_loss=%.4f)", seed, stopped_epoch_s, seed_runs[-1]["min_train_loss"])

df_seed_runs = pd.DataFrame([{k: v for k, v in r.items() if k != "model"} for r in seed_runs])
df_seed_runs

## Bagian 6 - Evaluasi Ensemble + Perbandingan vs Naive Tanpa Bobot (Notebook 13)

In [ ]:
ensemble_models = [r["model"] for r in seed_runs]
with torch.no_grad():
    ens_val_probs = np.mean([torch.softmax(m(X_val_t), dim=1)[:, 1].cpu().numpy() for m in ensemble_models], axis=0)
    ens_test_probs = np.mean([torch.softmax(m(X_test_t), dim=1)[:, 1].cpu().numpy() for m in ensemble_models], axis=0)

ensemble_threshold, ensemble_val_f1_macro = sweep_best_threshold(y_val, ens_val_probs)
y_test_pred = (ens_test_probs >= ensemble_threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test_arr, y_test_pred, labels=[0, 1]).ravel()

metrics_w = {
    "accuracy": accuracy_score(y_test_arr, y_test_pred),
    "precision": precision_score(y_test_arr, y_test_pred, zero_division=0),
    "recall": recall_score(y_test_arr, y_test_pred, zero_division=0),
    "f1": f1_score(y_test_arr, y_test_pred, zero_division=0),
    "f1_macro": f1_score(y_test_arr, y_test_pred, average="macro", zero_division=0),
}
diagnostics_w = {
    "auc_test": roc_auc_score(y_test_arr, ens_test_probs) if len(set(y_test_arr)) > 1 else float("nan"),
    "bal_acc_test": balanced_accuracy_score(y_test_arr, y_test_pred),
    "std_prob_up": float(ens_test_probs.std()),
    "min_train_loss": float(min(r["min_train_loss"] for r in seed_runs)),
}
collapse_w = bool(tn == 0 or tp == 0 or diagnostics_w["std_prob_up"] < 0.02)

print("=== Notebook 17 - Naive + Sample Weighting (ensemble 6 seed, best_config BARU) ===")
print(f"threshold (dari val, ensemble) : {ensemble_threshold:.3f}")
for k, v in metrics_w.items():
    print(f"  {k:10s}: {v:.4f}")
print("Diagnostik:")
for k, v in diagnostics_w.items():
    print(f"  {k:14s}: {v:.4f}")
print("Confusion matrix:", {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)})
print("Collapse:", collapse_w)

naive_metrics, naive_diag, naive_cm = naive_metadata["metrics"], naive_metadata["diagnostics"], naive_metadata["confusion_matrix"]
naive_collapse = bool(naive_cm["tn"] == 0 or naive_cm["tp"] == 0 or naive_diag["std_prob_up"] < 0.02)

comparison = pd.DataFrame({
    "Naive TANPA bobot (Notebook 13)": {
        **naive_metrics, **naive_diag,
        "n_train": naive_metadata["split_sizes"]["train_samples"],
        "n_test": naive_metadata["split_sizes"]["test_samples"],
        "collapse": naive_collapse,
    },
    "Naive + Sample Weighting (HPO baru)": {
        **metrics_w, **diagnostics_w,
        "n_train": len(X_train), "n_test": len(X_test), "collapse": collapse_w,
    },
})
comparison_path = OUT_DIR / "naive_vs_naive_weighted_comparison.csv"
comparison.to_csv(comparison_path)
print(f"\nDisimpan ke: {comparison_path}")
comparison

## Bagian 7 - Diagnosis Cepat: Apakah `train_loss` Membaik?

Uji kapasitas singkat (tanpa regularisasi/early stopping) memakai
`best_config` BARU + loss berbobot, untuk memastikan penurunan `train_loss`
(jika ada) bukan sekadar model yang lebih besar/lebih lama dilatih, tapi
representasi loss yang memang lebih mudah dioptimalkan.

In [ ]:
torch.manual_seed(SEED)
model_cap = FlexibleLSTMClassifier(
    input_size=len(FEATURE_COLS), hidden_size=best_config["hidden_size"],
    num_layers=best_config["num_layers"], dropout=0.0, activation=best_config["activation"], num_classes=2,
).to(device)
optimizer_cap = torch.optim.Adam(model_cap.parameters(), lr=best_config["lr"], weight_decay=0.0)
class_weights_cap = compute_class_weights(y_train, device)
train_loader_cap = DataLoader(train_ds, batch_size=best_config["batch_size"], shuffle=True)

CAPACITY_EPOCHS = 150
cap_losses = []
for epoch in range(1, CAPACITY_EPOCHS + 1):
    tl, _ = train_one_epoch_weighted(model_cap, train_loader_cap, optimizer_cap, class_weights_cap, device)
    cap_losses.append(tl)
    if epoch == 1 or epoch % 50 == 0 or epoch == CAPACITY_EPOCHS:
        logger.info("  [uji kapasitas weighted] epoch %3d/%d - train_loss=%.4f", epoch, CAPACITY_EPOCHS, tl)

print(f"\nmin_train_loss (uji kapasitas, {CAPACITY_EPOCHS} epoch, weighted, no early stop): {min(cap_losses):.4f}")
print(f"(pembanding) min_train_loss ensemble Bagian 6 (early-stopped)       : {diagnostics_w['min_train_loss']:.4f}")
print(f"(pembanding) min_train_loss Notebook 13 (naive TANPA bobot)         : {naive_diag['min_train_loss']:.4f}")

## Penutup

Perbandingan Bagian 6 murni karena PERUBAHAN skema loss (sample weighting)
+ Random Search baru - sengaja TIDAK reuse `best_config` lama supaya
konfigurasi memang dioptimalkan untuk loss yang baru, bukan konfigurasi
lama yang dipaksakan. Notebook 13 TIDAK diubah oleh notebook ini.